# 02 - Model Selection and Training
### DI Leather · Forecasting weekly production output (`actual_qty`)

| | |
|---|---|
| **Input** | `final_features.csv` (from Notebook 1) |
| **Output** | `production_model.joblib` (the full preprocessing + model pipeline) |
| **Next notebook** | `03_Prediction.ipynb` |

**Part A - Model selection:** train seven regression models on the earlier 80% of weeks, test them on the most recent 20%, and pick the best.

**Part B - Final training:** retrain the chosen model on all weeks and save it.


---
# Part A - Model Selection
## Step 1 - Imports

In [1]:
# DataFrame.to_markdown() needs the 'tabulate' package:  pip install tabulate

import warnings
import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, TargetEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from xgboost import XGBRegressor

warnings.filterwarnings('ignore')

## Step 2 - Load the feature table
The file must already be in chronological order; the check below stops the notebook if it is not.

In [2]:
df = pd.read_csv("final_features.csv", parse_dates=['week_start_date'])

assert df['week_start_date'].is_monotonic_increasing, "Data must be sorted by week_start_date"

print("Dataset shape:", df.shape)

Dataset shape: (9495, 19)


## Step 3 - Features, target and chronological 80/20 split
- **Target:** `actual_qty`.
- **Features:** every column except the target and the `week_start_date` ordering key.
- **Split:** the first 80% of rows (earlier weeks) train the model; the last 20% (most recent weeks) test it. There is **no random shuffling** - a random split would let the model see future weeks during training and give over-optimistic results.


In [3]:
target = 'actual_qty'

X_full = df.drop(columns=[target, 'week_start_date'])
y_full = df[target]

split_idx = int(len(df) * 0.8)
X_train, X_test = X_full.iloc[:split_idx], X_full.iloc[split_idx:]
y_train, y_test = y_full.iloc[:split_idx], y_full.iloc[split_idx:]

print(f"Train: {len(X_train):,} rows  ({df['week_start_date'].iloc[0].date()} to {df['week_start_date'].iloc[split_idx - 1].date()})")
print(f"Test : {len(X_test):,} rows  ({df['week_start_date'].iloc[split_idx].date()} to {df['week_start_date'].iloc[-1].date()})")

Train: 7,596 rows  (2022-04-18 to 2025-03-24)
Test : 1,899 rows  (2025-03-24 to 2026-03-30)


**How to read this:** the test period must start *after* the training period ends. The model is therefore judged on weeks it has never seen, the same situation it faces when forecasting real future production.


## Step 4 - Preprocessing pipeline
Each group of columns is transformed differently. The transformers live **inside** the pipeline, so they are fitted on training rows only when the pipeline is trained.

| Transformer | Columns | What it does |
|---|---|---|
| `StandardScaler` | numeric columns | Rescales each to mean 0, standard deviation 1 |
| `OneHotEncoder` | `financial_month`, `factory_unit`, `product_family`, `grade`, `order_category`, `size_id` | One 0/1 column per category; unseen categories become all zeros |
| `TargetEncoder` | `item_name` | Replaces each item by a smoothed average of `actual_qty` for that item; unseen items get the overall average |


In [4]:
categorical_ohe = ['financial_month', 'factory_unit', 'product_family', 'grade', 'order_category', 'size_id']
categorical_target = ['item_name']
numerical_cols = [col for col in X_train.columns if col not in categorical_ohe + categorical_target]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat_ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_ohe),
        ('cat_te', TargetEncoder(target_type='continuous'), categorical_target)
    ])

print("Numeric columns:", numerical_cols)

Numeric columns: ['week_no', 'week_of_month', 'rate', 'no_of_workers', 'working_hours', 'total_duration_per_shoe_mins', 'machine_count', 'theoretical_max_production', 'machine_worker_ratio', 'avg_hours_per_worker']


## Step 5 - Candidate models
Seven regressors covering three families:

| Family | Models |
|---|---|
| Linear | Linear Regression, Ridge, Lasso |
| Bagged trees | Random Forest, Extra Trees |
| Boosted trees | Gradient Boosting, XGBoost |


In [5]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(),
    'Lasso Regression': Lasso(alpha=0.1),
    'Random Forest': RandomForestRegressor(random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42),
    'Extra Trees': ExtraTreesRegressor(random_state=42, n_jobs=-1),
    'XGBoost': XGBRegressor(random_state=42, n_jobs=-1)
}

## Step 6 - Train and evaluate every model
Each model is wrapped in the same pipeline (preprocessing + model), trained on the training weeks, and scored on **both** the training and the test weeks. Comparing the two reveals overfitting.

Metrics: **R²** (share of variation explained, higher is better), **MAE** (average error in shoes) and **RMSE** (error in shoes, penalising large misses more).

In [6]:
results = []

for name, model in models.items():
    pipeline = Pipeline(steps=[('preprocessor', clone(preprocessor)), ('model', model)])

    pipeline.fit(X_train, y_train)

    y_train_pred = pipeline.predict(X_train)
    y_test_pred = pipeline.predict(X_test)

    results.append({
        'Model': name,
        'Train R²': round(r2_score(y_train, y_train_pred), 3),
        'Test R²': round(r2_score(y_test, y_test_pred), 3),
        'Train MAE': round(mean_absolute_error(y_train, y_train_pred), 3),
        'Test MAE': round(mean_absolute_error(y_test, y_test_pred), 3),
        'Train RMSE': round(np.sqrt(mean_squared_error(y_train, y_train_pred)), 3),
        'Test RMSE': round(np.sqrt(mean_squared_error(y_test, y_test_pred)), 3)
    })

results_df = pd.DataFrame(results)
print(results_df.to_markdown(index=False))

| Model             |   Train R² |   Test R² |   Train MAE |   Test MAE |   Train RMSE |   Test RMSE |
|:------------------|-----------:|----------:|------------:|-----------:|-------------:|------------:|
| Linear Regression |      0.891 |     0.887 |       3.371 |      3.482 |        4.297 |       4.42  |
| Ridge Regression  |      0.891 |     0.887 |       3.37  |      3.473 |        4.296 |       4.412 |
| Lasso Regression  |      0.887 |     0.882 |       3.427 |      3.539 |        4.374 |       4.511 |
| Random Forest     |      0.978 |     0.884 |       1.485 |      3.491 |        1.939 |       4.471 |
| Gradient Boosting |      0.907 |     0.89  |       3.13  |      3.404 |        3.984 |       4.347 |
| Extra Trees       |      0.999 |     0.88  |       0.18  |      3.573 |        0.333 |       4.544 |
| XGBoost           |      0.96  |     0.874 |       1.985 |      3.63  |        2.6   |       4.661 |


### Reading the results
- **Linear models** (Linear, Ridge, Lasso): train and test R² are almost identical (≈0.89 vs ≈0.88–0.89), so they do not overfit, and they already reach a test R² close to the best model. The inputs therefore relate to output largely in an additive way.
- **Random Forest, Extra Trees, XGBoost** overfit: train R² is 0.96–0.999 but test R² falls to 0.87–0.88. Extra Trees is the extreme case (0.999 → 0.881) - it nearly memorises the training weeks.
- **Gradient Boosting** has the **highest test R² (0.890)**, the **lowest test MAE (3.41 shoes)** and the **lowest test RMSE (4.35 shoes)**, with only a small train-test gap (0.906 → 0.890).
- **Practical size of the error:** an MAE of about 3.4 shoes against an average weekly row of roughly 37 shoes (from the EDA) is an error of about 9%.


## Step 7 - Select the best model
The model with the highest **test R²** is chosen.

In [7]:
best = results_df.loc[results_df['Test R²'].idxmax()]

print("Best model :", best['Model'])
print("Test R²    :", best['Test R²'])
print("Test MAE   :", best['Test MAE'])
print("Test RMSE  :", best['Test RMSE'])

Best model : Gradient Boosting
Test R²    : 0.89
Test MAE   : 3.404
Test RMSE  : 4.347


### Why Gradient Boosting - and why not the others
| Model | Decision | Reason |
|---|---|---|
| **Gradient Boosting** | **Selected** | Best test R², MAE and RMSE; small train-test gap, so it generalises to unseen weeks. |
| Linear / Ridge / Lasso | Rejected | Stable, but slightly lower test accuracy (R² 0.882–0.887, MAE 3.47–3.54). |
| Random Forest | Rejected | Overfits (train R² 0.978 vs test 0.884) and has higher test error. |
| Extra Trees | Rejected | Severe overfitting (train R² 0.999 vs test 0.881). |
| XGBoost | Rejected | Overfits (train 0.962 vs test 0.874) and has the **lowest** test R² of all seven models. |

Note that the selection margin is small (test R² 0.890 vs 0.887 for the best linear model), and the models were compared with default settings, without hyperparameter tuning.


---
# Part B - Final Training and Saving
## Step 8 -Retrain the selected model on all weeks
Part A used 80% of the weeks only so that the model could be tested honestly. For the model that will actually be used, it is retrained on **all** weeks, so it also learns from the most recent production patterns. The test scores from Part A remain the best estimate of its accuracy - a model trained on all weeks cannot be tested on data it has not seen.

The saved file contains the **whole pipeline** (preprocessing + model), so new data can be passed in directly.

In [8]:
final_pipeline = Pipeline(steps=[
    ('preprocessor', clone(preprocessor)),
    ('model', GradientBoostingRegressor(random_state=42))
])

print("Training final model...")
final_pipeline.fit(X_full, y_full)

model_filename = 'production_model.joblib'
joblib.dump(final_pipeline, model_filename)
print(f"Model successfully saved to {model_filename}")

Training final model...
Model successfully saved to production_model.joblib


## Step 9 - Check that the saved model loads and predicts
The file is reloaded and used on the last five rows. These weeks were part of the training data, so this only confirms the saved file works — it is **not** an accuracy test.

In [9]:
loaded_model = joblib.load('production_model.joblib')

sample = X_full.tail(5)
pd.DataFrame({
    'actual_qty': y_full.tail(5).values,
    'predicted_qty': loaded_model.predict(sample).round().astype(int)
})

,actual_qty,predicted_qty
0,27,24
1,22,26
2,33,31
3,25,25
4,24,25


---
## Summary
- Selected model: **Gradient Boosting** (test R² 0.890, test MAE 3.41 shoes, test RMSE 4.35 shoes).
- Saved pipeline: `production_model.joblib` - used by `03_Prediction.ipynb`.
